# 00 — Bridge Map Builder
## Canonical `bridges_map.csv` from PostgreSQL

This notebook rebuilds the Plan A / Plan B bridge-map dataset from the canonical PostgreSQL layers instead of relying on an old copied CSV.

### Source
- `Final_Project`
- `final.bridge` — canonical bridge/GIS/master attributes
- `final.traffic` — canonical bridge-level traffic features

### Outputs
- PostgreSQL: `final.bridge_map`
- CSV: `Dataset_PlanA-B/Map/bridges_map.csv`
- Parquet: `Dataset_PlanA-B/Map/bridges_map.parquet`
- Data inventory
- Data manifest

The generated map file is an export artifact. It is not a new ML training dataset and does not modify the frozen models.


## 01 — DATA SOURCE / TRANSFER MANIFEST

```text
PostgreSQL Final_Project
        │
        ├── final.bridge
        │      └── bridge / GIS / condition attributes
        │
        └── final.traffic
               └── traffic_dtv_latest / traffic_dtv_mean
                       ↓
             Notebook 00 — Bridge Map Builder
                       ↓
              final.bridge_map
                       │
                       ├── bridges_map.csv
                       └── bridges_map.parquet
```

`final.bridge` and `final.traffic` are the authoritative upstream sources.

`bridges_map.csv` is generated fresh from those sources. No previous `Repository/bridges_map.csv` is used.


In [1]:
# 02 — Portable project and PostgreSQL configuration

from pathlib import Path
from getpass import getpass
import os
import json
import hashlib
import pandas as pd
from sqlalchemy import create_engine, URL, text

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
MAP_DIR = DATASET_ROOT / "Map"
MAP_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILE = MAP_DIR / "bridges_map.csv"
PARQUET_FILE = MAP_DIR / "bridges_map.parquet"
MANIFEST_FILE = MAP_DIR / "00_bridge_map_data_manifest.txt"
INVENTORY_FILE = MAP_DIR / "00_bridge_map_data_inventory.csv"

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(url, connect_args={"connect_timeout": 10})

BASE_TABLE = '"final"."bridge"'
TRAFFIC_TABLE = '"final"."traffic"'
MAP_TABLE = '"final"."bridge_map"'

print("Project root:", PROJECT_ROOT)
print("Database:", DB_NAME)
print("Bridge source:", BASE_TABLE)
print("Traffic source:", TRAFFIC_TABLE)
print("Map DB table:", MAP_TABLE)
print("CSV:", CSV_FILE)


Project root: C:\Datenanalyse\final Project
Database: Final_Project
Bridge source: "final"."bridge"
Traffic source: "final"."traffic"
Map DB table: "final"."bridge_map"
CSV: C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\bridges_map.csv


In [2]:
# 03 — Source-table gate and schema inspection

required_tables = [
    ("final", "bridge"),
    ("final", "traffic"),
]

with engine.connect() as conn:
    for schema_name, table_name in required_tables:
        exists = conn.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.tables
                    WHERE table_schema = :schema_name
                      AND table_name = :table_name
                )
            """),
            {"schema_name": schema_name, "table_name": table_name},
        ).scalar()

        if not exists:
            raise RuntimeError(
                f"Required PostgreSQL table missing: {schema_name}.{table_name}"
            )

bridge_schema = pd.read_sql(
    text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'final'
          AND table_name = 'bridge'
        ORDER BY ordinal_position
    """),
    engine,
)

traffic_schema = pd.read_sql(
    text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'final'
          AND table_name = 'traffic'
        ORDER BY ordinal_position
    """),
    engine,
)

print("[PASS] Required source tables exist.")
print("final.bridge columns:", len(bridge_schema))
print("final.traffic columns:", len(traffic_schema))


[PASS] Required source tables exist.
final.bridge columns: 26
final.traffic columns: 21


In [3]:
# 04 — Resolve canonical source columns

bridge_cols = set(bridge_schema["column_name"].astype(str))
traffic_cols = set(traffic_schema["column_name"].astype(str))

def resolve_required(columns, candidates, label):
    for c in candidates:
        if c in columns:
            return c
    raise KeyError(
        f"Could not resolve required {label}. Candidates: {candidates}"
    )

def resolve_optional(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

BRIDGE_ID = resolve_required(
    bridge_cols,
    ["id_nr", "bridge_id"],
    "bridge identifier",
)

BRIDGE_TYPE = resolve_required(
    bridge_cols,
    ["bauwerksart_text", "bauwerksart", "bridge_type"],
    "bridge type",
)

MATERIAL = resolve_required(
    bridge_cols,
    ["baustoffklasse", "bauwerkstoff", "bridge_material", "material"],
    "material",
)

CONDITION = resolve_required(
    bridge_cols,
    ["zustandsnote", "condition_score"],
    "condition score",
)

CONDITION_CLASS = resolve_required(
    bridge_cols,
    ["zustandsnotenklasse", "condition_class"],
    "condition class",
)

BUILD_YEAR = resolve_optional(bridge_cols, ["baujahr", "build_year"])
LENGTH = resolve_optional(bridge_cols, ["laenge", "length"])
WIDTH = resolve_optional(bridge_cols, ["breite", "width"])
AREA = resolve_optional(bridge_cols, ["flaeche", "area"])

GIS_ORIGIN = resolve_optional(bridge_cols, ["gis_ort", "ort"])
GIS_DISTRICT = resolve_optional(bridge_cols, ["gis_kreis", "kreis"])
GIS_STATE = resolve_optional(bridge_cols, ["gis_bundesland", "bundesland"])

GEOM_X = resolve_optional(bridge_cols, ["geom_x"])
GEOM_Y = resolve_optional(bridge_cols, ["geom_y"])

TRAFFIC_BRIDGE_ID = resolve_required(
    traffic_cols,
    ["bridge_id", "id_nr"],
    "traffic bridge identifier",
)

DTV_LATEST = resolve_optional(
    traffic_cols,
    ["traffic_dtv_latest", "dtv_latest", "DTV_latest", "DTV"],
)

DTV_MEAN = resolve_optional(
    traffic_cols,
    ["traffic_dtv_mean", "dtv_mean", "DTV_mean"],
)

resolved = {
    "bridge_id": BRIDGE_ID,
    "bauwerksart_text": BRIDGE_TYPE,
    "baustoffklasse": MATERIAL,
    "zustandsnote": CONDITION,
    "zustandsnotenklasse": CONDITION_CLASS,
    "baujahr": BUILD_YEAR,
    "laenge": LENGTH,
    "breite": WIDTH,
    "flaeche": AREA,
    "gis_ort": GIS_ORIGIN,
    "gis_kreis": GIS_DISTRICT,
    "gis_bundesland": GIS_STATE,
    "geom_x": GEOM_X,
    "geom_y": GEOM_Y,
    "traffic_dtv_latest": DTV_LATEST,
    "traffic_dtv_mean": DTV_MEAN,
}

display(pd.DataFrame(
    [{"map_column": k, "source_column": v}
     for k, v in resolved.items()]
))

assert BRIDGE_ID is not None
assert BRIDGE_TYPE is not None
assert MATERIAL is not None
assert CONDITION is not None
assert CONDITION_CLASS is not None
assert TRAFFIC_BRIDGE_ID is not None


,map_column,source_column
0,bridge_id,id_nr
1,bauwerksart_text,bauwerksart_text
2,baustoffklasse,baustoffklasse
3,zustandsnote,zustandsnote
4,zustandsnotenklasse,zustandsnotenklasse
5,baujahr,baujahr
6,laenge,laenge
7,breite,breite
8,flaeche,flaeche
9,gis_ort,gis_ort


In [4]:
# 05 — Read bridge and traffic source data from PostgreSQL

bridge_select = {
    "bridge_id": f'"{BRIDGE_ID}"',
    "bauwerksart_text": f'"{BRIDGE_TYPE}"',
    "baustoffklasse": f'"{MATERIAL}"',
    "zustandsnote": f'"{CONDITION}"',
    "zustandsnotenklasse": f'"{CONDITION_CLASS}"',
}

for output_name, source_name in [
    ("baujahr", BUILD_YEAR),
    ("laenge", LENGTH),
    ("breite", WIDTH),
    ("flaeche", AREA),
    ("gis_ort", GIS_ORIGIN),
    ("gis_kreis", GIS_DISTRICT),
    ("gis_bundesland", GIS_STATE),
    ("geom_x", GEOM_X),
    ("geom_y", GEOM_Y),
]:
    if source_name:
        bridge_select[output_name] = f'"{source_name}"'
    else:
        bridge_select[output_name] = "NULL"

bridge_sql = """
SELECT
    {columns}
FROM "final"."bridge"
"""

bridge_sql = bridge_sql.format(
    columns=",\n    ".join(
        f"{expr} AS \"{name}\""
        for name, expr in bridge_select.items()
    )
)

bridge_df = pd.read_sql(text(bridge_sql), engine)

traffic_select = {
    "bridge_id": f'"{TRAFFIC_BRIDGE_ID}"',
    "traffic_dtv_latest": f'"{DTV_LATEST}"' if DTV_LATEST else "NULL",
    "traffic_dtv_mean": f'"{DTV_MEAN}"' if DTV_MEAN else "NULL",
}

traffic_sql = """
SELECT
    {columns}
FROM "final"."traffic"
"""

traffic_sql = traffic_sql.format(
    columns=",\n    ".join(
        f"{expr} AS \"{name}\""
        for name, expr in traffic_select.items()
    )
)

traffic_df = pd.read_sql(text(traffic_sql), engine)

print("Bridge rows:", len(bridge_df))
print("Traffic rows:", len(traffic_df))


Bridge rows: 52214
Traffic rows: 31290


In [5]:
# 06 — Controlled bridge-level merge

bridge_df["bridge_id"] = bridge_df["bridge_id"].astype(str).str.strip()
traffic_df["bridge_id"] = traffic_df["bridge_id"].astype(str).str.strip()

if bridge_df["bridge_id"].duplicated().any():
    raise RuntimeError("final.bridge does not have a unique bridge identifier.")

traffic_df = traffic_df.drop_duplicates(
    subset=["bridge_id"],
    keep="last",
)

bridge_map = bridge_df.merge(
    traffic_df,
    on="bridge_id",
    how="left",
    validate="one_to_one",
)

# Numeric normalization
for c in [
    "baujahr", "laenge", "breite", "flaeche",
    "zustandsnote", "geom_x", "geom_y",
    "traffic_dtv_latest", "traffic_dtv_mean",
]:
    if c in bridge_map.columns:
        bridge_map[c] = pd.to_numeric(
            bridge_map[c], errors="coerce"
        )

# Text normalization
for c in [
    "bauwerksart_text", "baustoffklasse",
    "zustandsnotenklasse", "gis_ort",
    "gis_kreis", "gis_bundesland",
]:
    if c in bridge_map.columns:
        bridge_map[c] = (
            bridge_map[c].astype("string").str.strip()
        )

bridge_map = bridge_map.reset_index(drop=True)
bridge_map.insert(0, "index", bridge_map.index)

print("Map rows:", len(bridge_map))
print("Map columns:", len(bridge_map.columns))
print("Unique bridge IDs:", bridge_map["bridge_id"].nunique())

assert len(bridge_map) == bridge_map["bridge_id"].nunique()


Map rows: 52214
Map columns: 17
Unique bridge IDs: 52214


In [6]:
# 07 — Validate the canonical map dataset

required_output = [
    "index",
    "bridge_id",
    "bauwerksart_text",
    "baujahr",
    "laenge",
    "breite",
    "flaeche",
    "baustoffklasse",
    "zustandsnote",
    "zustandsnotenklasse",
    "gis_ort",
    "gis_kreis",
    "gis_bundesland",
    "geom_x",
    "geom_y",
    "traffic_dtv_latest",
    "traffic_dtv_mean",
]

missing_output = [
    c for c in required_output
    if c not in bridge_map.columns
]

if missing_output:
    raise RuntimeError(
        f"Bridge-map output is missing required columns: {missing_output}"
    )

print("[PASS] bridges_map schema validated.")
print("Rows:", len(bridge_map))
print("Columns:", len(bridge_map.columns))
print("Condition missing %:",
      round(bridge_map["zustandsnote"].isna().mean() * 100, 2))
print("Geometry X missing %:",
      round(bridge_map["geom_x"].isna().mean() * 100, 2))
print("Geometry Y missing %:",
      round(bridge_map["geom_y"].isna().mean() * 100, 2))


[PASS] bridges_map schema validated.
Rows: 52214
Columns: 17
Condition missing %: 0.0
Geometry X missing %: 1.51
Geometry Y missing %: 1.51


In [7]:
# 08 — Write canonical PostgreSQL map table and local exports

with engine.begin() as conn:
    conn.execute(text('DROP TABLE IF EXISTS "final"."bridge_map"'))

bridge_map.to_sql(
    "bridge_map",
    engine,
    schema="final",
    if_exists="replace",
    index=False,
)

bridge_map.to_csv(
    CSV_FILE,
    index=False,
    encoding="utf-8-sig",
)

bridge_map.to_parquet(
    PARQUET_FILE,
    index=False,
)

print("[PASS] PostgreSQL table:", MAP_TABLE)
print("[PASS] CSV:", CSV_FILE)
print("[PASS] Parquet:", PARQUET_FILE)


[PASS] PostgreSQL table: "final"."bridge_map"
[PASS] CSV: C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\bridges_map.csv
[PASS] Parquet: C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\bridges_map.parquet


In [8]:
# 09 — Data inventory and manifest

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

artifacts = [
    ("postgresql_final_bridge", "Final_Project.final.bridge", "source"),
    ("postgresql_final_traffic", "Final_Project.final.traffic", "source"),
    ("postgresql_final_bridge_map", MAP_TABLE, "canonical generated map table"),
    ("bridges_map_csv", CSV_FILE, "canonical local map export"),
    ("bridges_map_parquet", PARQUET_FILE, "local map cache/export"),
]

inventory_rows = []

for name, location, role in artifacts:
    path = Path(location)
    is_file = path.exists()

    inventory_rows.append({
        "artifact": name,
        "location": location,
        "role": role,
        "exists": is_file if path.suffix else True,
        "size_bytes": path.stat().st_size if is_file else None,
        "sha256": sha256(path) if is_file else None,
    })

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(
    INVENTORY_FILE,
    index=False,
    encoding="utf-8-sig",
)

MANIFEST_FILE.write_text(
    f"""Notebook 00 — Bridge Map Builder
=================================

DATABASE:
  Final_Project

SOURCE TABLES:
  final.bridge
  final.traffic

GENERATED POSTGRESQL TABLE:
  final.bridge_map

CANONICAL LOCAL CSV:
  {CSV_FILE}

LOCAL PARQUET:
  {PARQUET_FILE}

DATA GRAIN:
  one row per bridge_id

ROW COUNT:
  {len(bridge_map)}

COLUMN COUNT:
  {len(bridge_map.columns)}

TRAFFIC JOIN:
  final.traffic.bridge_id -> bridge map bridge_id

MODEL TRAINING:
  NO

MODEL RETRAINING:
  NO

FEM / STRUCTURAL DESIGN:
  NO

SOURCE OF TRUTH:
  PostgreSQL final.bridge + final.traffic

The CSV is regenerated from PostgreSQL and must not be manually edited.
""",
    encoding="utf-8",
)

print("Inventory:", INVENTORY_FILE)
print("Manifest:", MANIFEST_FILE)


Inventory: C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\00_bridge_map_data_inventory.csv
Manifest: C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\00_bridge_map_data_manifest.txt


In [9]:
# 10 — Final integrity check

db_count = pd.read_sql(
    text('SELECT COUNT(*) AS n FROM "final"."bridge_map"'),
    engine,
)["n"].iloc[0]

db_unique = pd.read_sql(
    text('SELECT COUNT(DISTINCT bridge_id) AS n FROM "final"."bridge_map"'),
    engine,
)["n"].iloc[0]

assert int(db_count) == len(bridge_map)
assert int(db_unique) == len(bridge_map)
assert CSV_FILE.exists()
assert PARQUET_FILE.exists()
assert INVENTORY_FILE.exists()
assert MANIFEST_FILE.exists()

print("==============================================")
print("00 STATUS: COMPLETE")
print("==============================================")
print("PostgreSQL table :", MAP_TABLE)
print("CSV              :", CSV_FILE)
print("Parquet          :", PARQUET_FILE)
print("Data inventory   :", INVENTORY_FILE)
print("Data manifest    :", MANIFEST_FILE)
print("Rows             :", len(bridge_map))
print("Unique bridges   :", bridge_map["bridge_id"].nunique())


00 STATUS: COMPLETE
PostgreSQL table : "final"."bridge_map"
CSV              : C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\bridges_map.csv
Parquet          : C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\bridges_map.parquet
Data inventory   : C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\00_bridge_map_data_inventory.csv
Data manifest    : C:\Datenanalyse\final Project\Dataset_PlanA-B\Map\00_bridge_map_data_manifest.txt
Rows             : 52214
Unique bridges   : 52214
